# MB1 v0.2.2 — Trusted-Source Continuity Repair (Mode A only)

Targeted model-free candidate preparation with strict PRE/POST context, cut margins, ORB continuity and final automatic screening. No semantic annotation, CLIP, VLM, learned inference or model download.

Required Kaggle inputs:
1. Raw AIC corpus: `/kaggle/input/datasets/nadkli/dataset-aic`
2. MB1 v0.2.1 pack: `/kaggle/input/datasets/irthn1311/triage-eg-mb1-v021-candidates`
3. MB1 v0.2 pack: `/kaggle/input/datasets/irthn1311/triage-eg-mb1-v02-candidatess`
4. MB1 v0.2 AI-QC: `/kaggle/input/datasets/irthn1311/mb1-v02-ai-qc-pass1-bundle`
5. RT2 benchmark: `/kaggle/input/datasets/irthn1311/triage-eg-rt2-ai-benchmark-bundle`

Repository cloning is the only operation that may require Internet; experiment data and ORB screening are offline. Output ZIP: `/kaggle/working/triage_eg_mb1_v022_candidates.zip`.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

REPO_URL = os.environ.get('AIC_REPO_URL', 'https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF = os.environ.get('AIC_REPO_REF', 'TRIAGEEG')
REPO_DIR = Path(os.environ.get('AIC_REPO_DIR', '/kaggle/working/AIC2026_TeamPTK_SGU'))
if not (REPO_DIR / 'src/triage_eg').is_dir():
    if REPO_DIR.exists(): raise RuntimeError(f'Incomplete repository directory: {REPO_DIR}')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, str(REPO_DIR / 'src'))
print({'resolved_repo': str(REPO_DIR), 'ref': REPO_REF, 'commit': commit})

In [ ]:
DATA_INPUT = Path(os.environ.get('AIC_DATA_ROOT', '/kaggle/input/datasets/nadkli/dataset-aic'))
V021_INPUT = Path(os.environ.get('AIC_MB1_V021_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-mb1-v021-candidates'))
V02_INPUT = Path(os.environ.get('AIC_MB1_V02_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-mb1-v02-candidatess'))
QC_INPUT = Path(os.environ.get('AIC_MB1_V02_QC_ROOT', '/kaggle/input/datasets/irthn1311/mb1-v02-ai-qc-pass1-bundle'))
RT2_INPUT = Path(os.environ.get('AIC_RT2_ROOT', '/kaggle/input/datasets/irthn1311/triage-eg-rt2-ai-benchmark-bundle'))
OUTPUT_ROOT = Path('/kaggle/working/triage_eg_mb1_v022_candidates')
ZIP_PATH = Path('/kaggle/working/triage_eg_mb1_v022_candidates.zip')
RESOLVED_ROOT = Path('/kaggle/working/triage_eg_mb1_v022_resolved_inputs')
print({'data': str(DATA_INPUT), 'v021': str(V021_INPUT), 'v02': str(V02_INPUT), 'qc': str(QC_INPUT), 'rt2': str(RT2_INPUT), 'output': str(OUTPUT_ROOT)})

In [ ]:
SEARCH_ROOT = Path('/kaggle/input')
MAX_DEPTH, MAX_DIRECTORIES, MAX_ZIPS = 6, 5000, 100

def bounded_directories(root):
    queue, visited = [(Path(root), 0)], 0
    while queue:
        current, depth = queue.pop(0)
        if not current.is_dir(): continue
        visited += 1
        if visited > MAX_DIRECTORIES: raise RuntimeError('Input discovery exceeded directory bound')
        yield current
        if depth < MAX_DEPTH:
            queue.extend((child, depth + 1) for child in sorted(current.iterdir()) if child.is_dir() and not child.is_symlink())

def valid(data, filename):
    try:
        value = json.loads(next(x for x in data.decode().splitlines() if x.strip())) if filename.endswith('.jsonl') else json.loads(data)
    except Exception: return False
    if filename.startswith('mb1_v021_seed'): return value.get('seed_class') == 'MB1_V02_USABLE_SEED'
    if filename.startswith('mb1_v021_candidate'): return str(value.get('candidate_id', '')).startswith('mb1v021_')
    if filename == 'candidate_selection_v021.json': return value.get('experiment') == 'MB1_V021'
    if filename == 'cut_guard_regression_audit.json': return value.get('audit_role') == 'POST_HOC_DIAGNOSTIC_NOT_OFFICIAL_GT'
    if filename == 'mb1_v02_candidate_manifest.jsonl': return str(value.get('candidate_id', '')).startswith('mb1v02_')
    if filename == 'mb1_v02_ai_qc_pass1.jsonl': return 'qc_status' in value
    if filename == 'rt2_ai_benchmark.jsonl': return str(value.get('query_id', '')).startswith('rt2_')
    return True

def resolve_artifact(root, filename):
    base = root if root.exists() else SEARCH_ROOT
    directories = list(bounded_directories(base))
    matches = sorted({(d / filename).resolve() for d in directories if (d / filename).is_file() and valid((d / filename).read_bytes(), filename)})
    if len(matches) == 1: return matches[0]
    if len(matches) > 1: raise RuntimeError(f'Ambiguous {filename}: {matches}')
    hits = []
    for archive_path in sorted({p.resolve() for d in directories for p in d.glob('*.zip')})[:MAX_ZIPS]:
        with ZipFile(archive_path) as archive:
            for member in archive.namelist():
                if Path(member).name == filename:
                    data = archive.read(member)
                    if valid(data, filename): hits.append((archive_path, member, data))
    if len(hits) != 1: raise RuntimeError(f'Expected one {filename}; direct={matches}, zip_hits={[(str(a), m) for a,m,_ in hits]}')
    RESOLVED_ROOT.mkdir(parents=True, exist_ok=True)
    target = RESOLVED_ROOT / filename
    target.write_bytes(hits[0][2])
    return target.resolve()

def resolve_dataset(root):
    candidates = sorted({d.resolve() for d in bounded_directories(root if root.exists() else SEARCH_ROOT) if any(x.is_dir() for x in d.glob('Videos_*'))})
    if len(candidates) != 1: raise RuntimeError(f'Expected one raw dataset root: {candidates}')
    return candidates[0]

In [ ]:
DATASET_ROOT = resolve_dataset(DATA_INPUT)
V021_SEEDS = resolve_artifact(V021_INPUT, 'mb1_v021_seed_manifest.jsonl')
V021_MANIFEST = resolve_artifact(V021_INPUT, 'mb1_v021_candidate_manifest.jsonl')
V021_DIAGNOSTICS = resolve_artifact(V021_INPUT, 'mb1_v021_candidate_diagnostics.jsonl')
V021_SELECTION = resolve_artifact(V021_INPUT, 'candidate_selection_v021.json')
V021_CUT_AUDIT = resolve_artifact(V021_INPUT, 'cut_guard_regression_audit.json')
OLD_V02_MANIFEST = resolve_artifact(V02_INPUT, 'mb1_v02_candidate_manifest.jsonl')
AI_QC = resolve_artifact(QC_INPUT, 'mb1_v02_ai_qc_pass1.jsonl')
RT2_BENCHMARK = resolve_artifact(RT2_INPUT, 'rt2_ai_benchmark.jsonl')
RESOLVED = {'dataset': DATASET_ROOT, 'v021_seeds': V021_SEEDS, 'v021_manifest': V021_MANIFEST, 'v021_diagnostics': V021_DIAGNOSTICS, 'v021_selection': V021_SELECTION, 'v021_cut_audit': V021_CUT_AUDIT, 'old_v02_manifest': OLD_V02_MANIFEST, 'ai_qc': AI_QC, 'rt2': RT2_BENCHMARK}
print(json.dumps({k: str(v) for k,v in RESOLVED.items()}, indent=2))

In [ ]:
from triage_eg.experiments.mb1_v022 import MB1V022Config, preflight_mb1_v022

if OUTPUT_ROOT.exists():
    if OUTPUT_ROOT.parent != Path('/kaggle/working'): raise RuntimeError(f'Refusing cleanup outside /kaggle/working: {OUTPUT_ROOT}')
    shutil.rmtree(OUTPUT_ROOT)
ZIP_PATH.unlink(missing_ok=True)
CONFIG = MB1V022Config(dataset_root=DATASET_ROOT, v021_seed_manifest_path=V021_SEEDS, v021_candidate_manifest_path=V021_MANIFEST, v021_candidate_diagnostics_path=V021_DIAGNOSTICS, v021_selection_path=V021_SELECTION, v021_cut_audit_path=V021_CUT_AUDIT, old_v02_candidate_manifest_path=OLD_V02_MANIFEST, ai_qc_path=AI_QC, rt2_benchmark_path=RT2_BENCHMARK, output_root=OUTPUT_ROOT, build_git_commit=commit)
PREFLIGHT = preflight_mb1_v022(CONFIG)
print(json.dumps(PREFLIGHT, indent=2))
assert PREFLIGHT['frozen_seed_count'] == 13
assert 10 <= PREFLIGHT['trusted_source_count'] <= 16
assert PREFLIGHT['model_inference_required'] is False

In [ ]:
from triage_eg.experiments.mb1_v022 import prepare_mb1_v022_candidates

RESULT = prepare_mb1_v022_candidates(CONFIG)
OLD = RESULT['old_qc_audit']
GEO = RESULT['v021_geometry_audit']
print('OLD-QC CONTINUITY AUDIT — POST-HOC, NOT GT')
print(json.dumps({k: OLD[k] for k in ('old_hard_cut_count', 'old_hard_cut_rejected', 'OLD_HARD_CUT_RECALL', 'old_usable_count', 'old_usable_falsely_rejected', 'OLD_USABLE_FALSE_VETO_RATE', 'EDGE_BUG_AFFECTED_OLD_CASES')}, indent=2))
print('V0.2.1 GEOMETRY AUDIT')
print(json.dumps({k: GEO[k] for k in ('candidate_count', 'proposal_center_at_window_edge', 'V021_REJECTED_BY_CONTEXT_GEOMETRY')}, indent=2))
print('Thresholds were fixed before these audits and were not auto-tuned.')

In [ ]:
SELECTION = RESULT['selection']
RUN = RESULT['run_manifest']
rows = [json.loads(line) for line in (OUTPUT_ROOT / 'mb1_v022_candidate_manifest.jsonl').read_text().splitlines() if line.strip()]
print(json.dumps({'trusted_sources': SELECTION['trusted_source_count'], 'raw_proposals': SELECTION['raw_proposals'], 'rejected': SELECTION['rejected'], 'retained_NEW': SELECTION['retained_NEW_candidates'], 'combined_pool': SELECTION['combined_potential_pool'], 'per_video': SELECTION['per_video_retained_counts'], 'runtime_seconds': RUN['performance']['total_ms']/1000}, indent=2))
assert all(row['auto_continuity_status'] == 'AUTO_CONTINUITY_SCREEN_PASS' for row in rows)
assert all(row['proposal_center_frame'] not in (row['window_start_frame'], row['window_end_frame']) for row in rows)
assert all(row['pre_context_seconds'] >= 1.25 and row['post_context_seconds'] >= 1.25 for row in rows)
assert all((OUTPUT_ROOT / row['overview_sheet_path']).is_file() and (OUTPUT_ROOT / row['dense_sheet_path']).is_file() for row in rows)
print('Structural mapping/context gates: PASS')
print('AUTO_CONTINUITY_SCREEN_PASS is only a prefilter; GPT-5.6 Sol QC is still required.')

In [ ]:
from IPython.display import Image, display

for row in rows[:3]:
    print(row['candidate_id'], row['video_id'], row['proposal_score'], row['continuity_quality'])
    display(Image(filename=str(OUTPUT_ROOT / row['overview_sheet_path'])))
montages = sorted((OUTPUT_ROOT / 'montages').glob('overview_montage_*.jpg'))
if montages: display(Image(filename=str(montages[0])))

In [ ]:
from triage_eg.experiments.mb1_v022 import create_mb1_v022_bundle

archive = create_mb1_v022_bundle(OUTPUT_ROOT, ZIP_PATH)
with ZipFile(archive) as stream:
    members = stream.namelist()
assert not any(name.endswith(('.mp4','.npy','.npz','.pt','.pth','.bin')) for name in members)
print('DOWNLOAD ZIP:', archive)
print('size_bytes:', archive.stat().st_size, 'members:', len(members))
print('MB1_V022_REAL_STATUS = COMPLETE')
print('MB1_V022_AI_QC_STATUS = WAITING_FOR_AI')
print('M3_IMPLEMENTATION_STATUS = NOT_STARTED')